# Data quality checks

Checks on the raw tables loaded into DuckDB, before any cleaning.

In [1]:
from pathlib import Path

import duckdb

DB_PATH = Path("../data/f1.duckdb")
con = duckdb.connect(str(DB_PATH), read_only=True)

con.sql("SHOW TABLES")

nameVARCHAR
raw_drivers
raw_laps
raw_pit
raw_race_control
raw_session_result
raw_sessions
raw_stints
raw_weather


In [2]:
con.sql(
    """
    SELECT
        year,
        count(*) AS races_in_calendar,
        count(*) FILTER (WHERE coalesce(is_cancelled, false)) AS cancelled,
        count(*) FILTER (
            WHERE NOT coalesce(is_cancelled, false)
              AND date_end::TIMESTAMPTZ < now()
        ) AS finished
    FROM raw_sessions
    GROUP BY year
    ORDER BY year
    """
)

yearBIGINT,races_in_calendarBIGINT,cancelledBIGINT,finishedBIGINT
2023,23,1,22
2024,24,0,24
2025,24,0,24
2026,25,2,19


In [3]:
con.sql(
    """
    SELECT
        s.year,
        count(DISTINCT l.session_key) AS races_with_laps
    FROM raw_laps AS l
    JOIN raw_sessions AS s USING (session_key)
    GROUP BY s.year
    ORDER BY s.year
    """
)

yearBIGINT,races_with_lapsBIGINT
2023,22
2024,24
2025,24
2026,19


In [4]:
con.sql(
    """
    WITH per_race AS (
        SELECT
            s.year,
            s.circuit_short_name,
            d.session_key,
            count(*) AS drivers
        FROM raw_drivers AS d
        JOIN raw_sessions AS s USING (session_key)
        GROUP BY s.year, s.circuit_short_name, d.session_key
    ),
    with_expected AS (
        SELECT
            *,
            max(drivers) OVER (PARTITION BY year) AS expected
        FROM per_race
    )
    SELECT *
    FROM with_expected
    WHERE drivers < expected
    ORDER BY year
    """
)

yearBIGINT,circuit_short_nameVARCHAR,session_keyBIGINT,driversBIGINT,expectedBIGINT
2023,Singapore,9165,19,20
2024,Melbourne,9488,19,20
2025,Catalunya,9971,19,20


## Findings

- **Calendar**: 96 races from 2023 to 2026, including 3 cancelled races and the 4 races of 2026 still to be held. All 89 finished races were downloaded (laps available for every one of them).
- **Drivers per race**: 20 per race until 2025, 22 in 2026 (11 teams). Three races have 19 drivers: Singapore 2023, Melbourne 2024 and Barcelona 2025, where one car did not take part. This is not missing data: the grid size must never be assumed to be fixed.